# LangChain: RAG & Q&A

## Outline
* مفهوم RAG
* ساخت knowledge base (loader → split → embed → store)
* 2-Step RAG با LCEL
* Agentic RAG با `create_agent`
* مقایسه دو روش


In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())


## ۱. مفهوم RAG

**RAG (Retrieval-Augmented Generation)** دو محدودیت LLM را حل می‌کند:
- **Context محدود**: نمی‌تواند کل یک corpus را یکجا بخواند
- **دانش ثابت**: training data در یک زمان freeze شده

راه‌حل: در زمان query، اطلاعات مرتبط رو از خارج fetch کن و به LLM بده.

```
User Query → Retriever → Relevant Docs → LLM → Answer
```


## ۲. ساخت Knowledge Base

In [3]:
# pip install langchain-openai langchain-community faiss-cpu

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Step 1: Load ──
# loader = CSVLoader(file_path="OutdoorClothingCatalog_1000.csv")
# docs = loader.load()

# برای دمو، چند document دستی می‌سازیم
from langchain_core.documents import Document

docs = [
    Document(page_content="پیراهن محافظ خورشید دارای استاندارد UPF 50+ است و ۹۸ درصد از اشعه‌های فرابنفش را مسدود می‌کند. قابل شستشو با ماشین لباسشویی.", metadata={"source": "catalog", "id": 1}),
    Document(page_content="مایوهای شنای ما بسیار راحت هستند و سریع خشک می‌شوند. موجود در ۶ رنگ.", metadata={"source": "catalog", "id": 2}),
    Document(page_content="چکمه‌های کوهنوردی ضدآب هستند و پشتیبانی عالی از مچ پا در مسیرهای پیاده‌روی ارائه می‌دهند.", metadata={"source": "catalog", "id": 3}),
    Document(page_content="کلاه محافظ خورشید اشعه‌های فرابنفش را مسدود می‌کند و لبه پهنی دارد. طراحی سبک.", metadata={"source": "catalog", "id": 4}),
    Document(page_content="ژاکت پشمی در هوای سرد گرما را حفظ می‌کند. قابل شستشو با ماشین لباسشویی و با قابلیت بسته‌بندی آسان.", metadata={"source": "catalog", "id": 5}),
]
print(f"Loaded {len(docs)} documents")
print(f"Sample: {docs[-1].page_content[:100]}")


Loaded 5 documents
Sample: ژاکت پشمی در هوای سرد گرما را حفظ می‌کند. قابل شستشو با ماشین لباسشویی و با قابلیت بسته‌بندی آسان.


C:\Users\Alireza\AppData\Local\Temp\ipykernel_23276\3235098230.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [5]:
# ── Step 2: Split ──
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
)
splits = splitter.split_documents(docs)
print(f"Split into {len(splits)} chunks")


Split into 5 chunks


In [7]:
# ── Step 3: Embed & Store ──
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# تست embedding
sample_embed = embeddings.embed_query("پیراهن محافظ خورشید")
print(f"Embedding dimension: {len(sample_embed)}")
print(f"First 5 values: {sample_embed[:5]}")


Embedding dimension: 3072
First 5 values: [-0.0455322265625, -0.04132080078125, 0.011566162109375, 0.01099395751953125, 0.04278564453125]


In [9]:
# ساخت vector store
vectorstore = FAISS.from_documents(splits, embeddings)
print("Vector store created!")

# تست similarity search
query = "پیراهن هایی با محافظت در برابر نور خورشید"
results = vectorstore.similarity_search(query, k=2)
print(f"\nTop {len(results)} results for '{query}':")
for i, doc in enumerate(results):
    print(f"  {i+1}. {doc.page_content[:100]}")


Vector store created!

Top 2 results for 'پیراهن هایی با محافظت در برابر نور خورشید':
  1. پیراهن محافظ خورشید دارای استاندارد UPF 50+ است و ۹۸ درصد از اشعه‌های فرابنفش را مسدود می‌کند. قابل 
  2. کلاه محافظ خورشید اشعه‌های فرابنفش را مسدود می‌کند و لبه پهنی دارد. طراحی سبک.


In [11]:
# تغییر متد برای گرفتن داکیومنت به همراه امتیاز شباهت
results = vectorstore.similarity_search_with_relevance_scores(query, k=2)
print(f"\nTop {len(results)} results for '{query}':")

# باز کردن جفت (doc, score) در حلقه
for i, (doc, score) in enumerate(results):
    percentage = score * 100  # تبدیل امتیاز به درصد
    print(f"  {i+1}. {doc.page_content[:100]} (Similarity: {percentage:.2f}%)")


Top 2 results for 'پیراهن هایی با محافظت در برابر نور خورشید':
  1. پیراهن محافظ خورشید دارای استاندارد UPF 50+ است و ۹۸ درصد از اشعه‌های فرابنفش را مسدود می‌کند. قابل  (Similarity: 59.96%)
  2. کلاه محافظ خورشید اشعه‌های فرابنفش را مسدود می‌کند و لبه پهنی دارد. طراحی سبک. (Similarity: 36.57%)


## ۳. 2-Step RAG با LCEL

## bad version

In [13]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer the question based only on the following context.


Context:
{context}"""),
("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)



# 2-Step RAG Chain
rag_chain = ( rag_prompt
            | llm
            | StrOutputParser()
)

query = "لطفاً تمام پیراهن های دارای محافظت در برابر خورشید را لیست کنید"
retriever_results = format_docs(vectorstore.similarity_search(query, k=2))

response = rag_chain.invoke({"context":retriever_results , "question": query})
print(response)


پیراهن محافظ خورشید با استاندارد UPF 50+ که ۹۸ درصد از اشعه‌های فرابنفش را مسدود می‌کند، قابل شستشو با ماشین لباسشویی است.


In [15]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer the question based only on the following context.
If you don't know the answer, say "I don't have that information."

Context:
{context}"""),
    ("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)



# 2-Step RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# تست
response = rag_chain.invoke("لطفاً تمام پیراهن های دارای محافظت در برابر خورشید را لیست کنید")
print(response)


پیراهن محافظ خورشید دارای استاندارد UPF 50+ است و ۹۸ درصد از اشعه‌های فرابنفش را مسدود می‌کند.


In [17]:
# streaming با RAG
print("Streaming RAG response:")
for chunk in rag_chain.stream("برای هوای سرد چه لباسی دارید؟"):
    print(chunk, end="", flush=True)
print()


Streaming RAG response:
برای هوای سرد، ژاکت پشمی مناسب است که گرما را حفظ می‌کند و قابل شستشو با ماشین لباسشویی است.


## ۴. Agentic RAG با `create_agent`

In [19]:
from langchain.agents import create_agent
from langchain.tools import tool

# RAG به عنوان tool
@tool
def search_catalog(query: str) -> str:
    """Search the product catalog for relevant items.
    Use this when the user asks about products, clothing, or outdoor gear."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant products found."
    return "\n".join([f"- {doc.page_content}" for doc in docs])

# ساخت RAG agent
rag_agent = create_agent(
    model=llm,
    tools=[search_catalog],
    system_prompt="""You are a helpful outdoor clothing store assistant.
Use the search_catalog tool to find relevant products before answering questions.
Always base your answers on the search results."""
)


In [21]:
response = rag_agent.invoke({
    "messages": [{"role": "user", "content": "چه محصولاتی برای محافظت در برابر نور خورشید دارید؟"}]
})
print(response["messages"][-1].content)


برای محافظت در برابر نور خورشید، محصولات زیر موجود هستند:

1. **کلاه محافظ خورشید**: این کلاه اشعه‌های فرابنفش را مسدود می‌کند و دارای لبه پهنی است. طراحی آن سبک و راحت است.

2. **پیراهن محافظ خورشید**: این پیراهن دارای استاندارد UPF 50+ است و ۹۸ درصد از اشعه‌های فرابنفش را مسدود می‌کند. همچنین قابل شستشو با ماشین لباسشویی است.

3. **ژاکت پشمی**: این ژاکت در هوای سرد گرما را حفظ می‌کند و قابلیت بسته‌بندی آسان دارد. همچنین قابل شستشو با ماشین لباسشویی است.

اگر به اطلاعات بیشتری نیاز دارید یا سوال دیگری دارید، خوشحال می‌شوم کمک کنم!


In [23]:
# streaming agent
print("Agent response:")
for chunk in rag_agent.stream(
    {"messages": [{"role": "user", "content": "من به چکمه کوهنوردی نیاز دارم. چه پیشنهادی دارید؟"}]},
    stream_mode="values"
):
    latest = chunk["messages"][-1]
    if hasattr(latest, "content") and latest.content:
        print(latest.content)


Agent response:
من به چکمه کوهنوردی نیاز دارم. چه پیشنهادی دارید؟
- چکمه‌های کوهنوردی ضدآب هستند و پشتیبانی عالی از مچ پا در مسیرهای پیاده‌روی ارائه می‌دهند.
- کلاه محافظ خورشید اشعه‌های فرابنفش را مسدود می‌کند و لبه پهنی دارد. طراحی سبک.
- ژاکت پشمی در هوای سرد گرما را حفظ می‌کند. قابل شستشو با ماشین لباسشویی و با قابلیت بسته‌بندی آسان.
من چند گزینه برای چکمه‌های کوهنوردی دارم که می‌توانند به شما کمک کنند:

1. **چکمه‌های کوهنوردی ضدآب**: این چکمه‌ها پشتیبانی عالی از مچ پا را در مسیرهای پیاده‌روی ارائه می‌دهند و برای شرایط مرطوب مناسب هستند.

اگر به اطلاعات بیشتری درباره این چکمه‌ها یا سایر محصولات نیاز دارید، لطفاً بفرمایید!


In [25]:
response = rag_agent.invoke({
    "messages": [{"role": "user", "content": "پایتخت ایران کجاست"}]
})
print(response["messages"][-1].content)


پایتخت ایران، تهران است.


## ۵. مقایسه 2-Step RAG vs Agentic RAG

| | 2-Step RAG | Agentic RAG |
|---|---|---|
| **روش** | همیشه retrieve، بعد generate | agent تصمیم می‌گیره کِی retrieve کنه |
| **سرعت** | سریع‌تر، قابل پیش‌بینی | کندتر، variable |
| **انعطاف** | کم — همیشه یه retrieval | زیاد — می‌تونه چند بار search کنه |
| **کنترل** | زیاد | کم‌تر |
| **مناسب برای** | FAQ، documentation bot | research assistant، multi-source |


## جمع‌بندی — نصب مورد نیاز

```bash
pip install langchain langchain-openai langchain-community faiss-cpu
```

**Pipeline اصلی RAG:**
1. **Load** — `CSVLoader`, `PyPDFLoader`, `WebBaseLoader`
2. **Split** — `RecursiveCharacterTextSplitter`
3. **Embed** — `OpenAIEmbeddings`
4. **Store** — `FAISS`, `Chroma`, `PineconeVectorStore`
5. **Retrieve** — `vectorstore.as_retriever()`
6. **Generate** — LCEL chain یا `create_agent`
